### Imports & Paths

In [40]:
import numpy as np
import pandas as pd
from pathlib import Path

BASE       = Path.cwd().parent.parent / "ansys_data"
OUTPUT_DIR = Path.cwd().parent.parent / "data" / "raw"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### Pipeline constants & normal CFD anchors

In [41]:

D_INNER  = 0.188                          # m
A_PIPE   = np.pi / 4 * D_INNER**2        # 0.027752 m²
RHO      = 1679.0                         # kg/m³  slurry mixture density
CD       = 0.61                           # sharp-edged rectangular orifice (standard)
V_INLET  = 3.0                            # m/s  (fixed velocity inlet BC — all scenarios)
Q_INLET  = V_INLET * A_PIPE              # m³/s

X_A, X_B, X_C = 5.0, 25.0, 45.0

# ── Normal CFD anchors (your validated run, t > 10 s) ─────────────────────────
# Reference: Pullum (2007) hydraulic gradient 1500–2000 Pa/m for thickened tailings
# CFD: 1908.3 Pa/m — within validated range
P_A_NORM = 76520.14   # Pa
P_B_NORM = 38311.03   # Pa
P_C_NORM =   189.49   # Pa
V_NORM   =   2.9499   # m/s
GRAD     = (P_A_NORM - P_C_NORM) / (X_C - X_A)   # 1908.3 Pa/m

# ── Normal transient shape (fitted to your CFD t=0 → t~7s decay) ─────────────
# Parametric: P_node(t) = P_ss + (P_peak - P_ss) * exp(-t / tau)
P_PEAK_A  = 117246.90   # Pa  at t=0.1s Node A
P_PEAK_B  =  58469.72   # Pa  at t=0.1s Node B
P_PEAK_C  =    359.57   # Pa  at t=0.1s Node C
TAU_BASE  =    1.80     # s   decay constant

# ── Time grid ─────────────────────────────────────────────────────────────────
N_STEPS = 701
DT      = 0.1
TIMES   = np.round(np.arange(N_STEPS) * DT, 1)

### Leak scenario configuration

In [42]:
# Hole areas — rectangular orifices (from ANSYS CFD report)
HOLE_AREAS = {
    "incipient": 0.010 * 0.010,   # 10×10 mm → 1e-4  m²
    "moderate":  0.020 * 0.020,   # 20×20 mm → 4e-4  m²
    "critical":  0.050 * 0.050,   # 50×50 mm → 25e-4 m²
}

# Leak axial locations (m) — from ANSYS CFD report
LEAK_POSITIONS = [10.0, 20.0, 35.0, 42.0]

# Runs per (severity × location) — 5 runs × 12 combos = 60 runs × 701 = 42,060 rows
N_RUNS = 6  # 15 runs × 4 locations × 701 = 42,060 rows per severity

LABEL = {"incipient": 1, "moderate": 1, "critical": 1}

In [43]:
def leak_steady_state(severity: str, x_leak: float) -> dict:
    """
    Compute steady-state pressure and velocity at nodes A, B, C
    for a given leak severity and location.

    Physics:
    - Inlet BC is fixed velocity (3 m/s) — upstream velocity unchanged
    - Leaked flow calculated via Torricelli at local pipe pressure
    - Downstream velocity drops by leaked fraction
    - Downstream pressures scale with (V_down/V_inlet)^2 per Darcy-Weisbach
    - Upstream pressures: tiny rise due to reduced downstream back-pressure
      (negative pressure wave propagates upstream — AIP Advances 2024)
    """
    A_hole = HOLE_AREAS[severity]

    # Pressure at leak location (linear gradient interpolation from normal)
    P_at_leak = max(P_A_NORM - GRAD * (x_leak - X_A), 0.0)

    # Orifice leak flow (Torricelli, Cd=0.61 sharp-edged rectangular)
    v_orifice = CD * np.sqrt(2.0 * P_at_leak / RHO)
    Q_leak    = A_hole * v_orifice
    Q_down    = max(Q_INLET - Q_leak, 0.0)
    frac_lost = Q_leak / Q_INLET
    V_down    = Q_down / A_PIPE

    # Flow ratio squared — scales downstream pressures (Darcy-Weisbach ∝ V²)
    flow_sq = (Q_down / Q_INLET) ** 2

    # ── Pressure at each node ──────────────────────────────────────────────────
    def node_pressure(x_node, p_norm):
        if x_node < x_leak:
            # Upstream: tiny pressure rise from negative pressure wave reflection
            # Adegboye (2021): upstream P sensitive to both size and location
            return p_norm * (1.0 + 0.008 * frac_lost)
        else:
            # Downstream: scales with flow ratio squared (Darcy-Weisbach)
            return p_norm * flow_sq

    pA = node_pressure(X_A, P_A_NORM)
    pB = node_pressure(X_B, P_B_NORM)
    pC = node_pressure(X_C, P_C_NORM)

    # ── Velocity at each node ──────────────────────────────────────────────────
    # Fixed velocity inlet → upstream nodes hold at V_INLET
    # Downstream nodes carry reduced flow V_down
    vA = V_INLET if X_A < x_leak else V_down
    vB = V_INLET if X_B < x_leak else V_down
    vC = V_INLET if X_C < x_leak else V_down

    return dict(pA=pA, pB=pB, pC=pC,
                vA=vA, vB=vB, vC=vC,
                frac_lost=frac_lost, V_down=V_down)


def build_transient(p_ss, p_peak, tau, rng):
    """
    Build pressure array for one node over 701 timesteps.
    Starts ~0, spikes at t=0.1s, exponential decay to steady state.
    Matches real ANSYS Fluent transient shape from your CFD run.
    """
    arr = np.empty(N_STEPS)
    noise_ss = max(abs(p_ss) * 1e-4, 0.5)   # tiny numerical noise, like CFD
    for i, t in enumerate(TIMES):
        if t == 0.0:
            arr[i] = 0.0
        else:
            decay = np.exp(-t / tau)
            arr[i] = (p_ss + rng.normal(0, noise_ss)
                      + (p_peak - p_ss) * decay)
    return arr

In [44]:
def generate_leak_run(severity: str, x_leak: float,
                      run_id: int, rng: np.random.Generator) -> pd.DataFrame:
    """Generate one complete 701-row leak scenario run."""

    ss = leak_steady_state(severity, x_leak)

    # Run-to-run variability: ±2% peak, ±5% tau — same as normal script
    peak_scale = rng.uniform(0.98, 1.02)
    tau        = TAU_BASE * rng.uniform(0.95, 1.05)
    # Slight global pump-speed jitter (±0.05%)
    gs         = rng.normal(1.0, 0.0005)

    pA = build_transient(ss["pA"] * gs,
                         P_PEAK_A * peak_scale * (ss["pA"] / P_A_NORM),
                         tau, rng)
    pB = build_transient(ss["pB"] * gs,
                         P_PEAK_B * peak_scale * (ss["pB"] / P_B_NORM),
                         tau, rng)
    pC = build_transient(ss["pC"] * gs,
                         P_PEAK_C * peak_scale * max(ss["pC"] / P_C_NORM, 0.01),
                         tau * 0.8, rng)

    # Velocity: flat with tiny numerical noise (CFD-like)
    def vel_arr(v_ss):
        arr = np.full(N_STEPS, v_ss * gs)
        arr += rng.normal(0, max(v_ss * 5e-5, 1e-4), N_STEPS)
        arr[0] = V_NORM * gs    # t=0 same as normal (pipe not yet pressurised)
        return arr

    vA = vel_arr(ss["vA"])
    vB = vel_arr(ss["vB"])
    vC = vel_arr(ss["vC"])

    scenario_tag = f"leak_{severity}"

    return pd.DataFrame({
        "run_id":     run_id,
        "timestep":   np.arange(N_STEPS),
        "time_s":     TIMES,
        "pressure_A": pA,
        "pressure_B": pB,
        "pressure_C": pC,
        "velocity_A": vA,
        "velocity_B": vB,
        "velocity_C": vC,
        "label":      LABEL[severity],
        "scenario":   scenario_tag,
    })


### Generate all runs & save

In [45]:
rng = np.random.default_rng(seed=42)
all_runs  = []
global_id = 0

for severity in ["incipient", "moderate", "critical"]:
    for x_leak in LEAK_POSITIONS:
        for _ in range(N_RUNS):
            all_runs.append(
                generate_leak_run(severity, x_leak, global_id, rng)
            )
            global_id += 1

leak_df = pd.concat(all_runs, ignore_index=True)
out_path = OUTPUT_DIR / "leak_combined.csv"
leak_df.to_csv(out_path, index=False, float_format="%.6f")

# ── Summary ───────────────────────────────────────────────────────────────────
print("=" * 65)
print(f"  Saved  →  {out_path}")
print(f"  Total rows    : {len(leak_df):,}")
print(f"  Total runs    : {global_id}  ({N_RUNS} per severity×location)")
print(f"  Scenarios     : {leak_df['scenario'].nunique()}")
print("=" * 65)
print(f"\n{'Scenario':<30} {'label':>5}  {'SS pA':>9}  {'SS pB':>9}  {'SS pC':>7}  {'SS vB':>7}  {'SS vC':>7}")
print("-" * 65)
ss_df = leak_df[leak_df["time_s"] >= 10]
for sc in leak_df["scenario"].unique():
    sub  = ss_df[ss_df["scenario"] == sc]
    r0   = sub[sub["run_id"] == sub["run_id"].min()]
    lbl  = leak_df[leak_df["scenario"] == sc]["label"].iloc[0]
    print(f"{sc:<30}  {lbl:>5}  "
          f"{r0['pressure_A'].mean():>9.1f}  "
          f"{r0['pressure_B'].mean():>9.1f}  "
          f"{r0['pressure_C'].mean():>7.2f}  "
          f"{r0['velocity_B'].mean():>7.4f}  "
          f"{r0['velocity_C'].mean():>7.4f}")
print("=" * 65)
print(f"\nNormal reference:  pA=76520  pB=38311  pC=189.49")

  Saved  →  /home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/raw/leak_combined.csv
  Total rows    : 50,472
  Total runs    : 72  (6 per severity×location)
  Scenarios     : 3

Scenario                       label      SS pA      SS pB    SS pC    SS vB    SS vC
-----------------------------------------------------------------
leak_incipient                      1    76557.4    37827.8   187.02   2.9815   2.9815
leak_moderate                       1    76521.5    36325.0   179.64   2.9207   2.9207
leak_critical                       1    76625.5    26804.9   132.55   2.5092   2.5093

Normal reference:  pA=76520  pB=38311  pC=189.49


In [46]:
leak_df.shape

(50472, 11)